# 👁️ Convolutional Neural Network (CNN) — Practice Notebook

**This notebook contains guided exercises — implement the # TODO blocks.**

**Difficulty**: ⭐⭐ Intermediate  
**Time**: ~60 minutes

---


## 🎯 Section 1: Overview

A **Convolutional Neural Network (CNN)** is a class of deep neural network most commonly applied to analyzing visual imagery. Unlike fully connected layers, CNN layers use **convolution operations** that share parameters across space, enabling shift-invariance and spatial hierarchies.

### Receptive Fields and Architecture
By stacking convolutional layers, the network learns increasingly complex features — from simple edges in early layers to full shapes and objects in deeper layers.


## 📐 Section 2: Math & Intuition

### Output Dimension Formula
Given an input image of size $W$, kernel size $K$, padding $P$, and stride $S$:
$$O = \left\lfloor \frac{W - K + 2P}{S} \right\rfloor + 1$$

### 2D Convolution Operation (Single Channel)
$$Y_{i, j} = \sum_{m=0}^{K_h - 1} \sum_{n=0}^{K_w - 1} X_{i \cdot S + m, j \cdot S + n} W_{m, n} + b$$

### Receptive Field Calculation
$$RF_l = RF_{l-1} + (K_l - 1) \cdot J_{l-1}$$
where $J_{l-1}$ is the jump/stride up to the previous layer: $J_{l-1} = \prod_{i=1}^{l-1} S_i$.


## 🔧 Section 3: Implementation from Scratch


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
print('CNN Setup complete! ✅')


### 3.1 2D Convolution Forward Pass in NumPy


In [ ]:
def conv2d_forward(X, W, b, stride=1, padding=0):
    """
    Forward pass for a 2D convolution layer (single channel).
    X: input matrix of shape (H, W)
    W: kernel weights of shape (Kh, Kw)
    b: bias (scalar)
    """
    H, W_in = X.shape
    Kh, Kw = W.shape
    
    # Apply padding
    if padding > 0:
        X_pad = np.pad(X, padding, mode='constant', constant_values=0)
    else:
        X_pad = X
        
    H_pad, W_pad = X_pad.shape
    
    # Compute output dimensions
    H_out = int((H_pad - Kh) / stride) + 1
    W_out = int((W_pad - Kw) / stride) + 1
    
    out = np.zeros((H_out, W_out))
    
    # TODO: Implement convolution slicing loop
    
    return out


### 3.2 Max Pooling Forward Pass in NumPy


In [ ]:
def maxpool_forward(X, pool_size=2, stride=2):
    """
    Forward pass for Max Pooling 2D layer.
    """
    H, W = X.shape
    H_out = int((H - pool_size) / stride) + 1
    W_out = int((W - pool_size) / stride) + 1
    
    out = np.zeros((H_out, W_out))
    
    # TODO: Implement Max Pooling slicing loop
            
    return out


### 3.3 Verify NumPy Implementations


In [ ]:
X_test = np.array([
    [1, 2, 3, 0],
    [0, 1, 2, 1],
    [2, 1, 1, 0],
    [0, 1, 0, 1]
])
W_test = np.array([
    [1, 0],
    [0, 1]
])
bias = 1.0

c_out = conv2d_forward(X_test, W_test, bias, stride=1, padding=0)
print('Conv Output:\n', c_out)
if 'TODO' not in conv2d_forward.__code__.co_consts:
    assert c_out.shape == (3, 3)
    assert c_out[0, 0] == 3.0  # (1*1 + 2*0 + 0*0 + 1*1) + 1
    print('Convolution verification passed! ✅')

p_out = maxpool_forward(X_test, pool_size=2, stride=2)
print('Pooling Output:\n', p_out)
if 'TODO' not in maxpool_forward.__code__.co_consts:
    assert p_out.shape == (2, 2)
    assert p_out[0, 0] == 2.0
    print('Max Pooling verification passed! ✅')


## 📦 Section 4: Library Implementation


We will build a convolutional neural network (CNN) in PyTorch to classify simulated simple image representations.


In [ ]:
import torch
import torch.nn as nn

class PyTorchCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: Define Conv2d -> ReLU -> MaxPool2d -> Flatten -> Linear
        
    def forward(self, x):
        # TODO: Implement forward pass
        


### 4.1 Verification on Dummy Batch


In [ ]:
model = PyTorchCNN()
dummy_batch = torch.randn(8, 1, 28, 28)  # batch size of 8, single-channel 28x28 images
outputs = model(dummy_batch)
print('Output shape (should be [8, 10]):', list(outputs.shape))
assert list(outputs.shape) == [8, 10]
print('PyTorch CNN compilation check successful! ✅')


## 🧪 Section 5: Experiments


Visualizing the output of a convolution filter reveals how it extracts edge features.


In [ ]:
# Generate dummy image with vertical edge
img = np.zeros((28, 28))
img[:, 14:] = 1.0

# Vertical edge filter
v_kernel = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1]
])

feat_map = conv2d_forward(img, v_kernel, 0.0, stride=1, padding=1)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(img, cmap='gray')
axes[0].set_title('Original Image')
axes[1].imshow(feat_map, cmap='coolwarm')
axes[1].set_title('Filtered Image (Edge Filter)')
plt.show()


## ❓ Section 6: Interview Questions


### Q1: How do CNNs achieve translational invariance and parameter efficiency?
**Answer**:
- **Parameter Efficiency**: In a convolutional layer, the same kernel weights are reused (slid) across the entire input grid. A 3x3 conv layer has just $9$ weights regardless of the image dimensions, compared to $H \times W$ weights for fully connected layers.
- **Translational Invariance**: Parameter sharing combined with Pooling (like Max Pooling) ensures that if a feature shifts slightly in position, the activation map still detects it, resulting in robust representations that are invariant to shifts.

### Q2: Write the formula for receptive field size.
**Answer**:
The receptive field $RF_l$ of layer $l$ is calculated recursively from input to output:
$$RF_l = RF_{l-1} + (K_l - 1) \cdot J_{l-1}$$
where $K_l$ is the kernel size of layer $l$, and $J_{l-1}$ is the cumulative stride of all preceding layers: $J_{l-1} = \prod_{i=1}^{l-1} S_i$.

### Q3: Calculate the parameter count of a convolutional layer.
**Answer**:
Given a layer with $C_{\text{in}}$ input channels, $C_{\text{out}}$ output channels, kernel dimensions $K_h \times K_w$, and bias enabled:
$$\text{Parameters} = C_{\text{out}} \times (C_{\text{in}} \times K_h \times K_w + 1)$$
For example, a layer with $3$ input channels, $16$ output channels, and a $3 \times 3$ kernel has: $16 \times (3 \times 3 \times 3 + 1) = 16 \times 28 = 448$ parameters.

### Q4: What is the difference between Dilated Convolution and Standard Convolution?
**Answer**:
Dilated convolutions introduce 'holes' (dilation rate $D$) in the kernel, skipping pixels. For a dilation rate $D=2$, the kernel elements are spaced 1 pixel apart. This increases the kernel's receptive field size without adding any parameter count or computational cost.


## 🏆 Section 7: Challenge — Conv2D Backward Pass


**Challenge**: Implement the gradient calculation of a 2D convolution layer with respect to its weights ($dW$).


In [ ]:
def conv2d_backward_W(X, dZ, W_shape):
    """
    Compute gradient of loss with respect to weights dW.
    X: Input feature map of shape (H, W)
    dZ: Gradient of loss with respect to layer output of shape (H_out, W_out)
    W_shape: tuple (Kh, Kw) representing kernel shape
    """
    Kh, Kw = W_shape
    dW = np.zeros(W_shape)
    
    # TODO: Implement dW computation
            
    return dW

# Check
X_val = np.ones((5, 5))
dZ_val = np.ones((3, 3))
dW = conv2d_backward_W(X_val, dZ_val, (3, 3))
print('dW matrix:\n', dW)
if 'TODO' not in conv2d_backward_W.__code__.co_consts:
    assert dW.shape == (3, 3)
    assert np.all(dW == 9.0)  # Each cell is sum of 3x3 ones (9.0)
